In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path


In [ ]:
url = "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv"

data_dir = Path("Victor")
if not data_dir.exists():
    data_dir = Path(".")

csv_file_name = "2026-07.csv"
csv_path = data_dir / csv_file_name

print("\nFase 1: Extraccion de datos")
if csv_path.exists():
    print(f"Archivo local encontrado: {csv_path}")
else:
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        csv_path.write_bytes(response.content)
        print(f"Descarga exitosa. Archivo guardado en: {csv_path}")
    except requests.exceptions.Timeout:
        raise TimeoutError("La solicitud excedio el tiempo de espera y no hay archivo local disponible.")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"No se pudo descargar el CSV y no hay archivo local disponible: {e}")


In [ ]:
print(f"Leyendo el archivo CSV {csv_path}...")
df_raw = pd.read_csv(csv_path)
print("Archivo CSV leido correctamente.")
print(f"Se cargaron {df_raw.shape[0]:,} filas y {df_raw.shape[1]:,} columnas.")
display(df_raw.head(80))


# Transformacion


In [ ]:
df = df_raw.copy()

df["datetime_retiro"] = pd.to_datetime(
    df["Fecha_Retiro"].astype(str) + " " + df["Hora_Retiro"].astype(str),
    dayfirst=True,
    errors="coerce",
)
df["datetime_arribo"] = pd.to_datetime(
    df["Fecha_Arribo"].astype(str) + " " + df["Hora_Arribo"].astype(str),
    dayfirst=True,
    errors="coerce",
)

df["duracion_min"] = (df["datetime_arribo"] - df["datetime_retiro"]).dt.total_seconds() / 60

df["Ciclo_Estacion_Retiro"] = df["Ciclo_Estacion_Retiro"].astype(str).str.strip()
df["Ciclo_EstacionArribo"] = df["Ciclo_EstacionArribo"].astype(str).str.strip()

print("Valores nulos despues de transformar fechas:")
display(df[["datetime_retiro", "datetime_arribo", "duracion_min"]].isna().sum().to_frame("nulos"))
display(df.head(80))
